# HRS Gold Layer DDL Master Functional Specification

---

# 1. Document Information

### 🟢 CONFIRM / 🔵 USER INPUT

This section identifies the specification document.

| Property           | Value                                       |
| ------------------ | ------------------------------------------- |
| Document Name      | HRS Gold Layer DDL Functional Specification – BMI Descriptive Statistics by Cohort, Wave, Race, and Gender |
| Version            | 1.0                                         |
| Author             | `<AUTHOR>`                                  |
| AI Assistant       | Claude                                      |
| Last Updated       | 2026-09-15                                  |
| Target Platform    | Databricks                                  |
| Compute            | Serverless                                  |
| Runtime            | 15.x                                        |
| SQL Dialect        | Databricks SQL / Spark SQL                  |
| Storage Format     | Delta                                       |
| Specification Type | DDL Only                                    |
| Target Data Layer  | Gold                                        |
| Primary Purpose    | Curated analytical and visualization data   |

**User Action**

Update:

* Author
* Version, if appropriate
* Last Updated
* Document Name, if appropriate

Confirm the remaining platform settings.

---

# 2. Template Instructions

This master template uses three categories to identify how each section should be handled.

| Category                | Meaning                                             | User Action        |
| ----------------------- | --------------------------------------------------- | ------------------ |
| 🔵 **USER INPUT**       | Information specific to the new analytical use case | Complete or modify |
| 🟢 **CONFIRM**          | Standard project value that may need to be changed  | Review and confirm |
| 🟡 **MASTER CONTROLLED** | Project-wide architectural requirement              | Do not modify      |

### Important

The user should **not modify every section of this specification**.

For a new Gold table:

1. Complete all 🔵 **USER INPUT** sections.
2. Review all 🟢 **CONFIRM** sections.
3. Leave all 🟡 **MASTER CONTROLLED** sections unchanged.

---

# 3. Purpose

### 🟡 MASTER CONTROLLED

This specification defines the requirements for generating a Databricks SQL DDL script for a **Gold-layer analytical table**.

Gold tables contain curated data derived primarily from the HRS Silver CDM and are designed for:

* Data analytics
* Business intelligence
* Reporting
* Dashboards
* Data visualization
* Statistical analysis
* Public health analysis
* Policy analysis

The generated DDL must create the physical Gold Delta table required by the analytical use case defined in this specification.

The Gold table should contain only the data elements required to support the defined analytical purpose.

---

# 4. Gold Layer Role

### 🟡 MASTER CONTROLLED

The Gold layer represents the **consumption and analytical layer** of the HRS data architecture.

```text
RAND HRS Data
      │
      ▼
Bronze Layer
Raw / Source Data
      │
      ▼
Silver CDM
Standardized Relational Data
      │
      ▼
Gold Layer
Curated Analytical Data
      │
      ├───────────────┐
      ▼               ▼
Analytics       Visualization
      │               │
      └───────┬───────┘
              ▼
       Policy Insights
```

Gold tables should therefore be designed around **analytical requirements**, rather than simply reproducing Silver CDM tables.

---

# 5. Analytical Use Case

### 🔵 USER INPUT

Every Gold table must have a clearly defined analytical purpose.

| Property          | Value                                     |
| ----------------- | ----------------------------------------- |
| Use Case ID       | `UC_DA_HRS_BMI_001`                       |
| Use Case Name     | HRS BMI Descriptive Statistics by Cohort, Wave, Race, and Gender |
| Business Question | How does BMI vary across HRS survey waves, cohorts, race, and gender groups? |
| Primary Users     | Public health / demographic data analysts, epidemiologists, policy analysts |
| Intended Use      | ANALYTICS / REPORTING                     |

## 5.1 Business Question

Describe the analytical question the Gold table is intended to answer.

> How does average BMI, and the spread of BMI values, differ across HRS survey waves, cohorts, race, and gender groups? This table provides pre-aggregated descriptive statistics (count, mean, standard deviation, minimum, and maximum) of BMI for each wave/cohort/race/gender combination, so analysts and visualization tools do not need to recompute these statistics from respondent-level data.

---

# 6. Gold Table Grain

### 🔵 USER INPUT

The table grain defines what **one row represents** in the Gold table.

The grain must be explicitly defined before the table structure is designed.

> One row represents one aggregated BMI descriptive-statistics summary for a unique combination of cohort, survey wave, race, and gender. This is an aggregate/summary grain, not a respondent-level grain — individual BMI observations are not stored in this table.

| Grain Component    | Description     |
| ------------------ | --------------- |
| Respondent         | N/A — table is pre-aggregated across respondents; no respondent-level identifier is stored |
| Wave               | One wave per row, via `wave_id` (FK to `dim_hrs_wave`) |
| Measures / Metrics | `bmi_count`, `bmi_mean`, `bmi_sd`, `bmi_min`, `bmi_max` computed for each `cohort_id` / `wave_id` / `raracem` / `ragender` combination |

### Grain Rule

The generated DDL must support the specified analytical grain.

**Note:** The source DDL explicitly excludes a raw `bmi` column so that no separate group is created per unique BMI value — rows are grouped only by wave/cohort/race/gender combinations.

---

# 7. Measures and Metrics

### 🔵 USER INPUT

Identify the analytical measures and metrics required by the use case.

### 7.1 Measures

A **measure** is a value that can be aggregated or used in an analytical calculation.

Examples:

* Respondent count
* Income
* Age
* Health score

| Measure | Description                                                                 | Data Type      |
| ------- | ---------------------------------------------------------------------------- | -------------- |
| `bmi`   | RAND HRS BMI (Body Mass Index), continuous variable. Used as the source measure for aggregation only — **not** persisted as a column in the Gold table (see Grain Rule note above). | DECIMAL(10,2) |

### 7.2 Metrics

A **metric** is a calculated analytical result produced using one or more measures.

Examples:

* Average age
* Median income
* Respondent percentage
* Rate by demographic group

| Metric      | Definition     |
| ----------- | -------------- |
| `bmi_count` | Total number of BMI records within a given cohort/wave/race/gender group |
| `bmi_mean`  | Mean (average) BMI within a given cohort/wave/race/gender group |
| `bmi_sd`    | Standard deviation of BMI within a given cohort/wave/race/gender group |
| `bmi_min`   | Minimum BMI value within a given cohort/wave/race/gender group |
| `bmi_max`   | Maximum BMI value within a given cohort/wave/race/gender group |

**User Action:** Define only the measures and metrics required by the use case.

---

# 8. Gold Table Type

### 🔵 USER INPUT

Select the appropriate analytical table type.

| Gold Table Type | Select |
| --------------- | -----: |
| Fact            |      ☑ |
| Dimension       |      ☐ |
| Aggregate       |      ☑ |
| Analytical      |      ☐ |
---
| Property           | Value       |
| ------------------ | ----------- |
| GOLD_TABLE_TYPE    | Aggregate Fact |
| ANALYTICAL_ROLE    | Pre-aggregated fact table storing BMI descriptive statistics (count, mean, standard deviation, min, max) at the cohort/wave/race/gender grain |
| ANALYTICAL_PURPOSE | Support public health and demographic analysis of BMI trends across HRS survey waves, cohorts, race, and gender without requiring respondent-level recomputation |

**User Action:** Select the table type and describe its analytical role.

---

# 9. Target Table Parameters

### 🔵 USER INPUT / 🟢 CONFIRM

Define the physical Gold table.

| Parameter                | Category      | Value           |
| ------------------------ | ------------- | --------------- |
| TARGET_TABLE_NAME        | 🔵 User Input | `fact_hrs_bmi_race_gender_stats` |
| TARGET_TABLE_DESCRIPTION | 🔵 User Input | BMI descriptive statistics table, grouped by cohort and wave |
| CATALOG_NAME             | 🟢 Confirm    | `dev_catalog`   |
| SCHEMA_NAME              | 🟢 Confirm    | `gld_star_hrs`  |
| STORAGE_FORMAT           | 🟢 Confirm    | `DELTA`         |
| TABLE_TYPE               | 🟢 Confirm    | `Managed Table` |

> **Note:** The source DDL script's header comment references a different target (`dev_catalog.gld_sdm_hrs.hrs_bmi_stats`), but the executable `CREATE TABLE` statement targets `dev_catalog.gld_star_hrs.fact_hrs_bmi_race_gender_stats`. This specification follows the executable statement as the source of truth. Recommend correcting the header comment in the DDL script for consistency.

**User Action**

The user must:

* Define the target table name.
* Define the target table description.
* Confirm the catalog.
* Confirm the schema.
* Confirm Delta storage.
* Confirm Managed Table.

---

# 10. Target Table Location

### 🟡 MASTER CONTROLLED

The fully qualified Gold table name is constructed as:

```text
<CATALOG_NAME>.<SCHEMA_NAME>.<TARGET_TABLE_NAME>
```

The DDL generator must use the values defined in Section 9.

The generator must not hard-code alternative catalog, schema, or table names.

---

# 11. Source Silver Tables

### 🔵 USER INPUT

Identify the Silver CDM tables required to support the Gold analytical table.

| Source Table | Purpose     |
| ------------ | ----------- |
| `<NOT SPECIFIED IN DDL>` | The source DDL script does not identify the Silver CDM table(s) that supply `raracem`, `ragender`, `hacohort`, and `bmi`. Per the "no invented requirements" rule, this must be supplied by the user rather than inferred. |

**User Action:** Identify all Silver tables required by the analytical use case. This is a missing requirement not present in the source DDL script and should be completed before this specification is considered final.

---

# 12. Gold Table Design

### 🔵 USER INPUT

Define how Silver data will be presented for analytical consumption.

Respondent-level BMI values are aggregated into group-level descriptive statistics rather than stored individually. The design intentionally excludes a raw `bmi` column so that the table does not create a separate group for each unique BMI value — rows are grouped only by `wave_id` / `cohort_id` / `raracem` / `ragender` combinations, with the aggregated statistics (`bmi_count`, `bmi_mean`, `bmi_sd`, `bmi_min`, `bmi_max`) stored per group.

* **Required dimensions:** `cohort_id`, `wave_id` (surrogate FKs to the star-schema dimension tables), plus the descriptive/business attributes `raracem`, `ragender`, `hacohort`
* **Required measures:** `bmi_count`, `bmi_mean`, `bmi_sd`, `bmi_min`, `bmi_max`
* **Required attributes:** race (`raracem`), gender (`ragender`), HRS cohort code (`hacohort`)
* **Required identifiers:** `fact_hrs_bmi_race_gender_stats_id` (surrogate key), `cohort_id`, `wave_id` (foreign keys)
* **Required derived analytical fields:** None — see Section 18

### Design Principles

The Gold table should contain only data required to support the defined analytical use case.

The Gold table should not simply reproduce an entire Silver CDM table unless that is explicitly required by the use case.

**User Action:** Define the analytical design.

---

# 13. Key Strategy

### 🔵 USER INPUT

Define the key requirements for the Gold table.

| Property                     | Value         |
| ---------------------------- | ------------- |
| Surrogate Key Required       | YES  |
| Surrogate Key Column         | `fact_hrs_bmi_race_gender_stats_id`    |
| Surrogate Key Type           | `BIGINT`      |
| Generated Always As Identity | YES  |
| Analytical Key               | `cohort_id` + `wave_id` + `raracem` + `ragender` (composite grain key)       |
| Business Key                 | N/A — no natural business key defined |

**User Action:** Define the key strategy based on the analytical grain.

---

# 14. Dimension Columns

### 🔵 USER INPUT

Define the dimensions required for analysis.

| Column       | Type     | Nullable | Description     |
| ------------ | -------- | -------- | --------------- |
| `cohort_id`  | BIGINT   | No       | Foreign key to `dim_hrs_cohort.cohort_id` |
| `wave_id`    | BIGINT   | No       | Foreign key to `dim_hrs_wave.wave_id` |
| `raracem`    | INT      | Yes      | RAND HRS race/ethnicity code |
| `ragender`   | INT      | Yes      | RAND HRS gender code |
| `hacohort`   | INT      | Yes      | HRS cohort code |

**User Action:** Add only dimensions required by the analytical use case.

---

# 15. Measure Columns

### 🔵 USER INPUT

Define the physical columns required to support analytical measures.

| Column       | Type   | Nullable | Description     |
| ------------ | ------ | -------- | --------------- |
| `bmi_count`  | INT    | Yes      | Total number of records for a BMI group |
| `bmi_mean`   | DOUBLE | Yes      | Mean BMI value for the group |
| `bmi_sd`     | DOUBLE | Yes      | Standard deviation of BMI for the group |
| `bmi_min`    | INT    | Yes      | Minimum BMI value for the group |
| `bmi_max`    | INT    | Yes      | Maximum BMI value for the group |

**Note:** The source DDL does not apply `NOT NULL` to these measure columns, so all are nullable.

---

# 16. Gold Column Definition Matrix

### 🔵 USER INPUT

This is the primary physical table-definition section.

Every physical Gold table column must be defined here.

| Column Name                          | Category  | Databricks Type | Nullable | Description     |
| ------------------------------------- | --------- | --------------- | -------- | --------------- |
| `fact_hrs_bmi_race_gender_stats_id`   | KEY       | BIGINT (GENERATED ALWAYS AS IDENTITY) | No | System-generated surrogate key |
| `cohort_id`                           | KEY       | BIGINT           | No       | Foreign key to `dim_hrs_cohort.cohort_id` |
| `wave_id`                             | KEY       | BIGINT           | No       | Foreign key to `dim_hrs_wave.wave_id` |
| `raracem`                             | DIMENSION | INT              | Yes      | RAND HRS race/ethnicity code |
| `ragender`                            | DIMENSION | INT              | Yes      | RAND HRS gender code |
| `hacohort`                            | DIMENSION | INT              | Yes      | HRS cohort code |
| `bmi_count`                           | MEASURE   | INT              | Yes      | Total number of records for a BMI group |
| `bmi_mean`                            | MEASURE   | DOUBLE           | Yes      | Mean BMI value for the group |
| `bmi_sd`                              | MEASURE   | DOUBLE           | Yes      | Standard deviation of BMI for the group |
| `bmi_min`                             | MEASURE   | INT              | Yes      | Minimum BMI value for the group |
| `bmi_max`                             | MEASURE   | INT              | Yes      | Maximum BMI value for the group |
| `create_date`                         | AUDIT     | DATE             | No       | Record creation date |
| `update_date`                         | AUDIT     | DATE             | No       | Last update date |
| `active`                              | AUDIT     | BOOLEAN          | No       | Active indicator |

**Note:** A raw `bmi` column (`DECIMAL(10,2)`) is intentionally excluded from the physical table — see the design note in Sections 6 and 12.

### Column Rule

The DDL generator must create **only the columns defined in this matrix**.

The generator must not invent additional columns.

---

# 17. Source-to-Gold Mapping

### 🔵 USER INPUT

Define source lineage for each applicable Gold column.

| Gold Column  | Silver Table         | Silver Column | HRS Source Variable | Transformation     |
| ------------ | --------------------- | -------------- | -------------------- | ------------------ |
| `raracem`    | `<NOT SPECIFIED IN DDL>` | `<NOT SPECIFIED IN DDL>` | `RARACEM` | Direct pass-through |
| `ragender`   | `<NOT SPECIFIED IN DDL>` | `<NOT SPECIFIED IN DDL>` | `RAGENDER` | Direct pass-through |
| `hacohort`   | `<NOT SPECIFIED IN DDL>` | `<NOT SPECIFIED IN DDL>` | `HACOHORT` | Direct pass-through |
| `bmi_count`  | `<NOT SPECIFIED IN DDL>` | `<NOT SPECIFIED IN DDL>` | `BMI` (RAND CONT variable) | `COUNT(bmi)` grouped by wave/cohort/race/gender |
| `bmi_mean`   | `<NOT SPECIFIED IN DDL>` | `<NOT SPECIFIED IN DDL>` | `BMI` | `AVG(bmi)` grouped by wave/cohort/race/gender |
| `bmi_sd`     | `<NOT SPECIFIED IN DDL>` | `<NOT SPECIFIED IN DDL>` | `BMI` | `STDDEV(bmi)` grouped by wave/cohort/race/gender |
| `bmi_min`    | `<NOT SPECIFIED IN DDL>` | `<NOT SPECIFIED IN DDL>` | `BMI` | `MIN(bmi)` grouped by wave/cohort/race/gender |
| `bmi_max`    | `<NOT SPECIFIED IN DDL>` | `<NOT SPECIFIED IN DDL>` | `BMI` | `MAX(bmi)` grouped by wave/cohort/race/gender |

**User Action:** The source DDL script does not identify the originating Silver CDM table(s) or Silver column names. These must be supplied by the user; they have not been inferred, per the "do not invent missing requirements" rule.

---

# 18. Derived Analytical Columns

### 🔵 USER INPUT / 🟢 CONFIRM

Define derived analytical attributes when applicable.

```text
Derived Analytical Columns Required: No
```

The source DDL contains no derived categorical columns (e.g., age_group-style buckets). The aggregated statistical columns (`bmi_count`, `bmi_mean`, `bmi_sd`, `bmi_min`, `bmi_max`) are treated as Measures (Section 15) rather than Derived Analytical Columns.

---

# 19. Audit Columns

### 🟢 CONFIRM

The standard Gold-layer audit columns are:

| Column      | Type    | Nullable | Description          |
| ----------- | ------- | -------- | -------------------- |
| create_date | DATE    | No       | Record creation date |
| update_date | DATE    | No       | Last update date     |
| active      | BOOLEAN | No       | Active indicator     |

**Confirmed:** The source DDL implements all three standard audit columns (`create_date`, `update_date`, `active`) exactly as specified above, all `NOT NULL`.

---

# 20. Table Constraints

### 🔵 USER INPUT / 🟢 CONFIRM

| Constraint  | Required     |
| ----------- | ------------ |
| PRIMARY KEY | YES (`fact_hrs_bmi_race_gender_stats_id`) |
| FOREIGN KEY | YES (`cohort_id`, `wave_id`) |
| UNIQUE      | NO |
| NOT NULL    | YES (`cohort_id`, `wave_id`, `create_date`, `update_date`, `active`) |
| CHECK       | NO |
| IDENTITY    | YES (`fact_hrs_bmi_race_gender_stats_id`, GENERATED ALWAYS AS IDENTITY) |

**User Action:** Define or confirm the constraints required by the Gold table design.

---

# 21. Foreign Key Relationships

### 🔵 USER INPUT

Define relationships to parent tables when applicable.

| Child Column | Parent Table                               | Parent Column |
| ------------ | ------------------------------------------- | ------------- |
| `cohort_id`  | `dev_catalog.gld_star_hrs.dim_hrs_cohort`   | `cohort_id`   |
| `wave_id`    | `dev_catalog.gld_star_hrs.dim_hrs_wave`     | `wave_id`     |

---

# 22. Table Comment

### 🔵 USER INPUT

Provide a concise business-oriented description of the Gold table.

> BMI descriptive statistics table. Grouped by Cohort and wave.

*(Table comment taken verbatim from the source DDL's `COMMENT` clause.)*

---

# 23. Column Comments

### 🔵 USER INPUT

Every Gold table column should have a meaningful business description.

Column comments should describe the business meaning of the column rather than simply repeating its technical name.

| Column                              | Comment (from source DDL) |
| ------------------------------------ | -------------------------- |
| `fact_hrs_bmi_race_gender_stats_id` | System-generated surrogate key |
| `cohort_id`                         | Foreign key to hrs_cohort_id *(note: DDL comment text references `hrs_cohort_id`; actual FK target is `dim_hrs_cohort.cohort_id` — recommend correcting this comment for clarity)* |
| `wave_id`                           | Foreign key to hrs_wave.wave_id |
| `raracem`                           | *(no comment in source DDL — recommend adding one, e.g., "RAND HRS race/ethnicity code")* |
| `ragender`                          | *(no comment in source DDL — recommend adding one, e.g., "RAND HRS gender code")* |
| `hacohort`                          | *(no comment in source DDL — recommend adding one, e.g., "HRS cohort code")* |
| `bmi_count`                         | Total number of records for a BMI value |
| `bmi_mean`                          | *(empty comment in source DDL — recommend adding one, e.g., "Mean BMI value for the group")* |
| `bmi_sd`                            | *(empty comment in source DDL — recommend adding one, e.g., "Standard deviation of BMI for the group")* |
| `bmi_min`                           | *(empty comment in source DDL — recommend adding one, e.g., "Minimum BMI value for the group")* |
| `bmi_max`                           | *(empty comment in source DDL — recommend adding one, e.g., "Maximum BMI value for the group")* |
| `create_date`                       | Record creation date |
| `update_date`                       | Last update date |
| `active`                            | Active indicator |

**User Action:** Several columns in the source DDL have missing or unclear comments; recommend updating the DDL script to add complete, business-meaningful comments for `raracem`, `ragender`, `hacohort`, `bmi_mean`, `bmi_sd`, `bmi_min`, `bmi_max`, and to correct the `cohort_id` comment.

---

# 24. Physical Table Requirements

### 🟡 MASTER CONTROLLED

Gold tables must conform to the standard HRS physical-table architecture.

Standard requirements include:

* Delta storage
* Managed tables
* Explicit column definitions
* Defined data types
* Defined nullability
* Appropriate keys and constraints
* Table comments
* Column comments

The generated DDL must conform to these requirements.

---

# 25. SQL Generation Requirements

### 🟡 MASTER CONTROLLED

The generated SQL must:

* Use Databricks SQL / Spark SQL.
* Use uppercase SQL keywords.
* Use consistent indentation.
* Use explicit column definitions.
* Use the specified catalog.
* Use the specified schema.
* Use the specified table name.
* Use Delta storage.
* Create a managed table.
* Include required comments.
* Include required constraints.
* Follow the completed specification exactly.

---

# 26. DROP TABLE Requirement

### 🟢 CONFIRM

| Requirement          | Value        |
| -------------------- | ------------ |
| DROP TABLE IF EXISTS | YES — present in source DDL |

**User Action:** Confirm whether the development process permits table recreation.

> **Warning:** `DROP TABLE` is destructive and should be used carefully.

---

# 27. Unsupported Objects

### 🟡 MASTER CONTROLLED

The DDL generator must not create unsupported or out-of-scope objects.

Do not generate:

```text
INSERT
UPDATE
DELETE
MERGE
SELECT
VIEWS
STORED PROCEDURES
FUNCTIONS
INDEXES
PARTITIONS
ZORDER
OPTIMIZE
```

DDL and DML responsibilities must remain separate.

---

# 28. DDL and DML Separation

### 🟡 MASTER CONTROLLED

The HRS Gold-layer development process separates table creation from table population.

```text
Gold DDL Specification
          │
          ▼
       Generate
          │
          ▼
      Gold DDL
          │
          ▼
 Execute in Databricks
          │
          ▼
   Create Gold Table
          │
          ▼
       Validate
          │
          ▼
Gold DML Specification
          │
          ▼
   Populate Gold Table
```

The DDL specification defines **what the table looks like**.

The DML specification defines **how the table is populated**.

---

# 29. Validation Requirements

### 🔵 USER INPUT / 🟢 CONFIRM

The following standard validations should be confirmed.

| Validation         | Required     |
| ------------------ | ------------ |
| Table Exists       | Yes          |
| Correct Catalog    | Yes          |
| Correct Schema     | Yes          |
| Correct Table Name | Yes          |
| Delta Format       | Yes          |
| Expected Columns   | Yes          |
| Data Types         | Yes          |
| Nullability        | Yes          |
| Primary Key        | Yes          |
| Foreign Keys       | Yes          |
| Unique Constraints | No           |

### Use-Case-Specific Validation

| Validation                                                                 | Required |
| ---------------------------------------------------------------------------- | -------- |
| No unique group per `cohort_id`/`wave_id`/`raracem`/`ragender` appears more than once | Yes |
| `bmi_count > 0` for every row (a group should not be materialized with zero underlying records) | Yes |
| `bmi_min <= bmi_mean <= bmi_max` for every row                              | Yes      |
| Raw `bmi` column is absent from the physical table                          | Yes      |

**User Action:** Confirm the standard validations and add analytical-use-case-specific validations.

---

# 30. Analytical Grain Validation

### 🔵 USER INPUT

Define how the analytical grain will be validated.

> One combination of cohort, wave, race, and gender must produce no more than one Gold table row.

SQL validation concept:

```text
COUNT(*) = COUNT(DISTINCT cohort_id, wave_id, raracem, ragender)
```

---

# 31. Gold DDL Workflow

### 🟡 MASTER CONTROLLED

The standard Gold DDL workflow is:

```text
Analytical Use Case
        │
        ▼
Business Question
        │
        ▼
Analytical Grain
        │
        ▼
Gold Table Type
        │
        ▼
Silver Source Tables
        │
        ▼
Dimensions + Measures
        │
        ▼
Gold Column Definitions
        │
        ▼
Keys + Constraints
        │
        ▼
DDL Specification
        │
        ▼
Generate DDL
        │
        ▼
Review SQL
        │
        ▼
Execute in Databricks
        │
        ▼
Validate Gold Table
        │
        ▼
Create Gold DML Specification
```

---

# 32. DDL Deliverable

### 🟡 MASTER CONTROLLED

The generated DDL should be saved using the standard repository convention:

```text
/sql/ddl/create_<TARGET_TABLE_NAME>.sql
```

The final AI-generated DDL deliverable must contain:

```text
SQL ONLY
```

No explanatory text should be included in the generated SQL deliverable.

---

# 33. AI DDL Generation Instructions

### 🟡 MASTER CONTROLLED

Generate a complete Databricks SQL DDL script from this specification.

The generated DDL must:

1. Follow the completed specification exactly.
2. Create only the specified Gold table.
3. Use the specified catalog, schema, and table name.
4. Create all specified columns.
5. Use the specified data types.
6. Apply the specified nullability.
7. Apply the specified keys and constraints.
8. Include required table comments.
9. Include required column comments.
10. Follow the specified SQL formatting requirements.
11. Use only DDL statements.
12. Not generate INSERT, UPDATE, DELETE, or MERGE statements.
13. Not generate unsupported database objects.
14. Not invent columns.
15. Not invent constraints.
16. Not invent business rules.
17. Not infer missing requirements.
18. Return SQL only.

If required information is missing from the completed specification, the missing requirement must be identified rather than invented.

---

# 34. Final DDL Specification Checklist

### 🟢 CONFIRM

Before submitting the specification for SQL generation, confirm:

* [x] Use Case defined
* [x] Business Question defined
* [x] Gold Table Type defined
* [x] Analytical Grain defined
* [x] Measures and Metrics defined
* [ ] Silver Source Tables identified — **NOT COMPLETE**: not specified in source DDL, requires user input
* [x] Dimensions defined
* [x] Measures defined
* [x] Gold Columns defined
* [ ] Source-to-Gold Mapping completed — **PARTIAL**: HRS source variables identified, Silver table/column names still required
* [x] Derived Columns defined or explicitly excluded (none required)
* [x] Key Strategy defined
* [x] Constraints defined
* [x] Foreign Keys defined or explicitly excluded
* [x] Table Comment defined
* [ ] Column Comments defined — **PARTIAL**: several columns have missing/empty comments in the source DDL (see Section 23)
* [x] Audit Columns confirmed
* [x] DROP TABLE requirement confirmed
* [x] Validation requirements defined
* [x] Analytical Grain Validation defined

---

# 35. Notebook Template

### 🟡 MASTER CONTROLLED

The master DDL specification is intended to be maintained and used within a Databricks notebook.

Recommended notebook structure:

```text
HRS_Gold_DDL_<TableName>.ipynb

    1. Document Information
    2. Purpose
    3. Gold Layer Role
    4. Analytical Use Case
    5. Gold Table Grain
    6. Measures and Metrics
    7. Gold Table Type
    8. Target Table Parameters
    9. Source Silver Tables
   10. Gold Table Design
   11. Key Strategy
   12. Dimension Columns
   13. Measure Columns
   14. Gold Column Definition Matrix
   15. Source-to-Gold Mapping
   16. Constraints
   17. Validation Requirements
   18. DDL Generation Instructions
```

---

# 36. Template Usage

### 🟡 MASTER CONTROLLED

To create a new Gold table:

### Step 1 — Start with the Master Template

Copy the HRS Gold Layer DDL Master Functional Specification.

### Step 2 — Complete User Input Sections

Complete all 🔵 **USER INPUT** sections.

### Step 3 — Review Confirm Sections

Review all 🟢 **CONFIRM** sections and change only where the analytical use case requires it.

### Step 4 — Leave Master-Controlled Sections Alone

Do not modify 🟡 **MASTER CONTROLLED** sections unless the HRS Gold architecture itself has changed.

### Step 5 — Complete the Checklist

Confirm that all required information has been provided.

### Step 6 — Generate the DDL

Provide the completed specification to the AI DDL generator.

### Step 7 — Review the Generated SQL

Verify that the SQL matches the specification.

### Step 8 — Execute in Databricks

Create the Gold Delta table.

### Step 9 — Validate the Table

Run the required structural and analytical validations.

### Step 10 — Create the DML Specification

Once the Gold table structure has been successfully validated, create the corresponding Gold DML specification.

---

# 37. Architecture Principles

### 🟡 MASTER CONTROLLED

The HRS Gold layer follows these architectural principles:

1. **Analytical Purpose First**
   Gold tables are designed around analytical requirements.

2. **Explicit Grain**
   Every Gold table must explicitly define what one row represents.

3. **Consumer-Oriented Design**
   Gold tables should be easy for analysts and visualization tools to consume.

4. **Purpose-Built Tables**
   A Gold table should contain only the data required by its analytical use case.

5. **Controlled Denormalization**
   Gold tables may combine information from multiple Silver CDM tables when required for analytical consumption.

6. **Traceable Lineage**
   Gold columns should have documented lineage back to Silver data and, where applicable, RAND HRS variables.

7. **Separation of Responsibilities**
   DDL defines the table structure. DML defines how the table is populated.

8. **Reusable Standards**
   The master template provides consistent standards across HRS analytical subject areas.

---

# 38. Overall HRS Data Architecture

### 🟡 MASTER CONTROLLED

The overall HRS data architecture is:

```text
                    RAND HRS Dataset
                           │
                           ▼
                    Bronze Layer
                    Raw HRS Data
                           │
                           ▼
                    Silver CDM
              Standardized HRS Data Model
                           │
             ┌─────────────┴─────────────┐
             ▼                           ▼
       Silver DDL                    Silver DML
       Table Design                 Data Loading
             │                           │
             └─────────────┬─────────────┘
                           ▼
                      Gold Layer
                Curated Analytical Tables
                           │
             ┌─────────────┼─────────────┐
             ▼             ▼             ▼
          Analytics   Visualization   Reporting
             │             │             │
             └─────────────┼─────────────┘
                           ▼
                    Policy Insights
```

---

# Appendix A — User Completion Summary

When creating a new Gold DDL specification, the user primarily works with the following sections.

## 🔵 USER INPUT — Complete

```text
5   Analytical Use Case
6   Gold Table Grain
7   Measures and Metrics
8   Gold Table Type
9   Target Table Parameters
11  Source Silver Tables
12  Gold Table Design
13  Key Strategy
14  Dimension Columns
15  Measure Columns
16  Gold Column Definition Matrix
17  Source-to-Gold Mapping
18  Derived Analytical Columns
20  Table Constraints
21  Foreign Key Relationships
22  Table Comment
23  Column Comments
29  Validation Requirements
30  Analytical Grain Validation
```

## 🟢 CONFIRM — Review

```text
1   Document Information
9   Target Table Parameters
18  Derived Analytical Columns
19  Audit Columns
20  Table Constraints
26  DROP TABLE Requirement
29  Validation Requirements
34  Final DDL Specification Checklist
```

## 🟡 MASTER CONTROLLED — Do Not Modify

```text
2   Template Instructions
3   Purpose
4   Gold Layer Role
10  Target Table Location
24  Physical Table Requirements
25  SQL Generation Requirements
27  Unsupported Objects
28  DDL and DML Separation
31  Gold DDL Workflow
32  DDL Deliverable
33  AI DDL Generation Instructions
35  Notebook Template
36  Template Usage
37  Architecture Principles
38  Overall HRS Data Architecture
```

---

# Appendix B — Quick Start

A new user can think of the entire template as answering ten questions:

```text
1. What analytical question are we answering?
                    │
                    ▼
2. What does one row represent?
                    │
                    ▼
3. What type of Gold table are we creating?
                    │
                    ▼
4. What Silver tables provide the data?
                    │
                    ▼
5. What dimensions do analysts need?
                    │
                    ▼
6. What measures do analysts need?
                    │
                    ▼
7. What columns must the Gold table contain?
                    │
                    ▼
8. What keys and constraints are required?
                    │
                    ▼
9. How will we validate the table?
                    │
                    ▼
10. Generate and validate the DDL
```

**The objective is not to make the user understand every architectural detail before starting. The objective is to guide the user through defining the analytical requirements while the master template controls the technical standards.**
